# M1.2 Electric Field by Superposition

Book pointer: chapter 2, sections 2.1 (Coulomb's law), 2.2 (electric field), 2.3 (visualizing fields), 2.4 (continuous distributions).

## 1. Recall

1. Write Coulomb's law for the force on $q_2$ from $q_1$ as a vector equation. Which way does the unit vector point?
2. Define the electric field. Why is it useful to separate the source from the test charge?
3. Write $\mathbf{E}(\mathbf{r})$ for $N$ point charges as a sum. Which vector appears in the numerator and what power of distance in the denominator?
4. Turn the sum into an integral for a volume charge density $\rho(\mathbf{r}')$. Which variable is integrated and which is held fixed?
5. Far from any finite charge distribution with net charge $Q \neq 0$, how does $|\mathbf{E}|$ fall off with distance?

## 2. Principle to algorithm

The field of one point charge at $\mathbf{r}_i$ observed at $\mathbf{r}$ is

$$\mathbf{E}_i(\mathbf{r}) = \frac{1}{4\pi\epsilon_0}\, \frac{q_i\,(\mathbf{r} - \mathbf{r}_i)}{|\mathbf{r} - \mathbf{r}_i|^3}.$$

Superposition says the total field is the plain sum. A continuous charge distribution is the same thing with many small charges: cut the body into pieces, give each piece the charge $\Delta q_i = \rho(\mathbf{r}_i)\Delta v$ at its centre, and sum. The error shrinks as the pieces shrink, as long as we do not observe from inside a piece.

The algorithm is therefore a double loop over observation points and source pieces, which NumPy does as a broadcast: an array of shape (N, 1, 3) minus one of shape (1, M, 3) gives all N by M displacement vectors at once. The library function `fields.point_charges_E` does this in chunks of sources so the memory stays bounded.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields

plt.rcParams["figure.figsize"] = (6, 5)
K = fields.K

In [ ]:
# A dipole: +q at (0, 0.5), -q at (0, -0.5). Field on a 2D grid in the plane z = 0.
q = 1e-9
src = np.array([[0, 0.5, 0], [0, -0.5, 0]])
chg = np.array([q, -q])

x, y, P = grids.grid2d((-2, 2), (-2, 2), 41)
E = fields.point_charges_E(src, chg, P)
X, Y, _ = grids.components(P)
Emag = grids.magnitude(E)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].quiver(X, Y, E[..., 0] / Emag, E[..., 1] / Emag, np.log10(Emag), cmap="viridis", pivot="mid")
axes[0].set_title("direction, coloured by log10 |E|")
# streamplot wants (ny, nx) arrays, so transpose the ij grid.
axes[1].streamplot(x, y, E[..., 0].T, E[..., 1].T, color=np.log10(Emag).T, cmap="viridis", density=1.4)
axes[1].set_title("field lines")
for ax in axes:
    ax.scatter(src[:, 0], src[:, 1], c=["red", "blue"], s=60, zorder=3); ax.set_aspect("equal")
plt.show()

### A continuous distribution: ring of charge

A ring of radius $R$ with total charge $Q$ is cut into $N$ equal point charges on the circle. The field on the axis has a closed form

$$E_z(z) = \frac{1}{4\pi\epsilon_0}\,\frac{Q\,z}{(z^2 + R^2)^{3/2}},$$

which is what we test against.

In [ ]:
def ring_sources(R, Q, N):
    """N equal point charges equally spaced on a circle of radius R in the plane z = 0."""
    phi = (np.arange(N) + 0.5) * 2 * np.pi / N
    pos = np.stack([R * np.cos(phi), R * np.sin(phi), np.zeros(N)], axis=-1)
    return pos, np.full(N, Q / N)

def ring_Ez_exact(z, R, Q):
    return K * Q * z / (z**2 + R**2) ** 1.5

R, Q = 1.0, 1e-9
z = np.linspace(-3, 3, 121)
axis = np.stack([0 * z, 0 * z, z], axis=-1)

pos, qs = ring_sources(R, Q, 64)
Ez = fields.point_charges_E(pos, qs, axis)[:, 2]

plt.plot(z, ring_Ez_exact(z, R, Q), "k-", lw=3, alpha=0.3, label="closed form")
plt.plot(z, Ez, "C1.", ms=4, label="N = 64 point charges")
plt.xlabel("z / R"); plt.ylabel("E_z  [V/m]"); plt.legend(); plt.show()

### Exercise: finite line of charge

A straight segment of length $2L$ along $x$, centred at the origin, carries total charge $Q$ uniformly. At perpendicular distance $d$ from its centre the field is

$$E_y(d) = \frac{1}{4\pi\epsilon_0}\,\frac{Q}{d\sqrt{d^2 + L^2}}.$$

Replace the body of `line_sources` so that it returns $N$ point charges along the segment. The check compares your discretisation to the closed form.

In [ ]:
def line_sources(L, Q, N):
    """N equal point charges on the segment x in [-L, L], y = z = 0."""
    xs = (np.arange(N) + 0.5) * (2 * L / N) - L          # replace with your own discretisation
    return np.stack([xs, 0 * xs, 0 * xs], axis=-1), np.full(N, Q / N)


L, d = 1.0, 0.7
pos, qs = line_sources(L, Q, 400)
Ey = fields.point_charges_E(pos, qs, [0.0, d, 0.0])[1]
Ey_exact = K * Q / (d * np.sqrt(d**2 + L**2))
print(f"numerical {Ey:.6e}   exact {Ey_exact:.6e}   relative error {abs(Ey / Ey_exact - 1):.2e}")
assert abs(Ey / Ey_exact - 1) < 1e-3

## 4. Verify

Convergence of the ring discretisation. The on-axis field is symmetric, so the error comes only from the finite number of pieces and should fall quickly with $N$. Off axis the field of a discrete ring differs from a continuous one close to the ring, so also look at a point in the plane of the ring at $r = 1.5R$. There the convergence is slower.

In [ ]:
Ns = np.array([4, 8, 16, 32, 64, 128, 256])
obs_axis = np.array([0.0, 0.0, 0.5])
obs_plane = np.array([1.5, 0.0, 0.0])
ref_plane = fields.point_charges_E(*ring_sources(R, Q, 20000), obs_plane)[0]

err_axis, err_plane = [], []
for N in Ns:
    pos, qs = ring_sources(R, Q, N)
    err_axis.append(abs(fields.point_charges_E(pos, qs, obs_axis)[2] / ring_Ez_exact(0.5, R, Q) - 1))
    err_plane.append(abs(fields.point_charges_E(pos, qs, obs_plane)[0] / ref_plane - 1))

plt.loglog(Ns, err_axis, "o-", label="on axis, z = 0.5 R (vs closed form)")
plt.loglog(Ns, np.maximum(err_plane, 1e-16), "s-", label="in plane, r = 1.5 R (vs N = 20000)")
plt.xlabel("number of pieces N"); plt.ylabel("relative error"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()
assert err_axis[-1] < 1e-6

## 5. Explore and rebuild

Explore. Build a uniformly charged square plate of side $a$ from an $n \times n$ grid of point charges. Compute $E_z$ on the axis from $z = 0.01a$ to $z = 100a$ and plot it on log-log axes. Identify the two limits: near the plate $E \to \sigma / 2\epsilon_0$ and far from it $E \to Q/(4\pi\epsilon_0 z^2)$. Where does the crossover happen relative to $a$? How does the near-field result change as you refine $n$?

Rebuild. Close this notebook and write `point_charges_E` from scratch so that it handles a single observation point, a list of points, and a full grid with one code path.